# Segmenting `grains.tif` with Ilastik and Python

This proposed solution consists of two steps; first, we separate (approximately) the grain bulks from the grain boundaries using [Ilastik](https://ilastik.org). 

<video width="320" height="240" controls>
  <source src="./ilastik-recording.mp4" type="video/mp4">
</video>

We export the `Simple Segmentation` output from Ilastik as a TIFF image.

Then, we read this segmentation mask into Python, and we apply a few simple operations to label the grains using [Scikit-image](https://scikit-image.org). These steps are described below.

In [ ]:
from pathlib import Path
import numpy as np

import skimage.io
from skimage.morphology import label
from skimage.segmentation import watershed
from scipy.ndimage import distance_transform_edt

In [ ]:
# Path to the `Simple Segmentation` output from Ilastik:
segmentation_file = "grains_Simple Segmentation.tif"

# Read the grain boundaries mask from Ilastik
segmentation = skimage.io.imread(segmentation_file)
segmentation = segmentation.astype(np.uint8)  # Make sure it's an 8-bit array

# Binary mask of the bulk (`background` class):
bulk = segmentation == 1

# Binary mask of the grain boundaries (`foreground` class)
boundaries = segmentation == 2

# Label the grain bulks (connected components):
labelled_bulk = label(bulk)

# Compute an euclidean distance transform in the grain boundary mask:
distance_img = distance_transform_edt(boundaries)

# Fill the remaining space (the grain boundaries) using a watershed transform:
grains_labelled = watershed(-distance_img, markers=labelled_bulk)

# Save the labelled image:
mask_path = Path(".").resolve().parents[1] / "Submissions" / "grains" / "example_solution.tif"

skimage.io.imsave(mask_path, grains_labelled)